In [ ]:
import os
import clip
import torch
from torchvision.transforms import AutoAugmentPolicy
from utils import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameter Configuration
EPOCH = 5
WEIGHT_DECAY = 1e-5

experiment_configurations = [
    {
        "name": "Merged",
        "lr": 1e-5,
        "batch_size": 4,
        "momentum": 0.95,
        "dataset": '../merged',
        "split": 0.7,
        "transform": transforms.Compose([
            transforms.Resize(224)
        ])
    },
    {
        "name": "baseline",
        "lr": 1e-5,
        "batch_size": 4,
        "momentum": 0.95,
        "dataset": '../dataset',
        "split": 0.7,
        "transform": transforms.Compose([
            transforms.Resize(224)
        ])
    },
    {
        "name": "RandomResizedCrop",
        "lr": 1e-5,
        "batch_size": 4,
        "momentum": 0.95,
        "dataset": '../dataset',
        "split": 0.7,
        "transform": transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.08, 1.0))
        ])
    },
    {
        "name": "AutoAugment",
        "lr": 1e-5,
        "batch_size": 4,
        "momentum": 0.95,
        "dataset": '../dataset',
        "split": 0.7,
        "transform": transforms.Compose([
            transforms.AutoAugment(policy=AutoAugmentPolicy.IMAGENET)
        ])
    },
    {
        "name": "Combined Geo Transforms",
        "lr": 1e-5,
        "batch_size": 4,
        "momentum": 0.95,
        "dataset": '../dataset',
        "split": 0.7,
        "transform": transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(degrees=(0, 360)),
            transforms.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0)),
        ])
    },
]

In [ ]:
def run_experiment(mn, save_dir, epochs, weight_decay, name, lr, batch_size, momentum, dataset, split, transform):
    print(f"Running {name} experiment with model name {mn} and is going to be saved in {save_dir}")
    #print(f"Running {name} experiment with lr={lr}, batch_size={batch_size}, momentum={momentum}, dataset={dataset}, split={split}, transform={transform}")
    model, preprocess = initialize_model(device, mn)
    dataset = LocalDataset(root_dir=dataset, preprocess=preprocess, transforms=transform)
    train_dataset, val_dataset = split_dataset(dataset, train_ratio=split)
    train_loader, val_loader = create_data_loaders(train_dataset, val_dataset, batch_size)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for epoch in range(epochs):
        train_model(model, train_loader, optimizer, device, epoch)
        val_accuracy = validate_model(model, val_loader, device)
    now = datetime.datetime.now()
    torch.save(model.state_dict(), f"5EPOCHS/{save_dir}/clip-{name}-{val_accuracy:.2f}.pth")

In [ ]:
model_names = ["RN50", "RN101", "ViT-B/16", "ViT-L/14"]
dir_model_names = ["RN50", "RN101", "ViT-B-16", "ViT-L-14"]

for i, mn in enumerate(model_names):
    for conf in experiment_configurations:
        run_experiment(mn, dir_model_names[i], EPOCH, WEIGHT_DECAY, **conf)

In [ ]:
for conf in experiment_configurations:
    run_experiment(EPOCH, WEIGHT_DECAY, **conf)

In [ ]:
run_experiment(EPOCH, WEIGHT_DECAY, **experiment_configurations[-1])